# Phase 2: model results and daily demand types

This notebook reads the saved reports and uses package charts. It performs no downloads or training. It reads the data folder the project is configured with (`SPAIN_ELECTRICITY_DATA_DIR`, or `data/` in the repository). Regenerate the reports with `scripts/train_models.py` after downloading and ingesting the history.

In [ ]:
import pandas as pd
from IPython.display import display

from spain_electricity import charts
from spain_electricity.config import DataPaths
from spain_electricity.model_results import comparison_table, saved_reports, update_table

paths = DataPaths.from_env()
reports = saved_reports(paths)
display(comparison_table(reports))

## Interpretation

Train: 2021–2024. Parameter selection: 2025. Each regression then refits the selected configuration on every valid 2021–2025 hour and is tested once on 2026. Demand is compared with REE's peninsular forecast and with the same hour 24 hours earlier; price and renewable share with the same hour 24 hours earlier. Every comparison uses the same paired hours. Negative electricity prices and renewable shares above one are valid.

## Does training on 2026 help?

The update refits the same configuration on every valid hour through October 2026 and is scored on the November 2026 hours, beside the 2021–2025 model on those same hours.

In [ ]:
display(update_table(reports))

In [ ]:
for name in ("demand", "price", "renewable_share"):
    report = reports.get(name, {})
    if report.get("features"):
        print(name)
        display(charts.importance_chart(pd.DataFrame(report["features"]), charts.LIGHT))

## Feature weights

Tree importances and standardized linear coefficients have different interpretations. Correlated predictors may share importance. Neither these weights nor the contemporaneous estimates establish causality or day-ahead forecasting skill.

## KMeans day types

KMeans selects k using training-day silhouette, then assigns other complete days to those centers. Each profile has mean one. Spring hour 2 is interpolated only on the known DST day, and the repeated autumn hour is averaged. KMeans numbers the groups arbitrarily; each group's name is inferred from its share of weekends or holidays and its mean temperature.

In [ ]:
if "day_types" in reports:
    report = reports["day_types"]
    display(pd.DataFrame([{"k": k, "silhouette": value} for k, value in report["silhouettes"].items()]))
    summary = pd.DataFrame(report["summary"])
    display(charts.profiles_chart(summary, charts.LIGHT))
    display(summary.drop(columns="mean_profile"))

## Next validation

Review the saved hours and units, compare each target with its baselines, and inspect errors before explaining their causes. Phase 3 is outside this notebook.